# Session 11 — Capstone Part 1: Real Data In

**Goal of this session:** download a real open fMRI dataset and pull out region time series.

*Python for Neuroscience, session 11 of 12.*

## Why this matters

Everything since session 5 has run on signals we invented. That was on purpose, because you can only check your method when you already know the answer.

Now we swap in real brains. The dataset is from a study where children and adults watched a short animated film inside an MRI scanner. It is open, it is small, and `nilearn` downloads it for you in a few seconds.

In [ ]:
# On Colab, install nilearn first. Locally, skip this if you already have it.
%pip install -q nilearn

## Getting the data

`fetch_development_fmri` handles the download and caches it, so running this twice is instant. We start with one participant.

Each participant gives you a preprocessed 4D image, meaning three spatial dimensions plus time, and a file of confound regressors: head motion estimates and similar nuisance signals that we want to remove.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from nilearn import datasets

dev = datasets.fetch_development_fmri(n_subjects=1)

print("functional image:", dev.func[0].split("/")[-1])
print("confounds file:  ", dev.confounds[0].split("/")[-1])

In [ ]:
import pandas as pd

phenotypic = pd.DataFrame(dev.phenotypic)
phenotypic[["participant_id", "Age", "Child_Adult"]]

## We need an atlas

A 4D image has hundreds of thousands of voxels, which is far too many to analyse directly and mostly noise anyway. An atlas divides the brain into a manageable number of labelled regions.

We use the MSDL atlas: 39 regions, each already assigned to a known functional network like the default mode or the visual system.

In [ ]:
import numpy as np

atlas = datasets.fetch_atlas_msdl()
labels = [str(x) for x in atlas.labels]
networks = [str(x) for x in atlas.networks]
coords = np.asarray(atlas.region_coords)

print(len(labels), "regions")
print("first six:", labels[:6])
print("networks: ", sorted(set(networks)))

## Where those regions are

Before we touch the signals, look at where they sit. Each dot is one atlas region, plotted on a glass brain seen from three angles.

In [ ]:
from nilearn import plotting

plotting.plot_markers(
    node_values=np.ones(len(coords)),
    node_coords=coords,
    node_size=40,
    display_mode="ortho",
    colorbar=False,
    title="MSDL atlas regions",
)
plotting.show()

## From voxels to time series

A masker does the reduction. It takes the 4D image, averages the voxels belonging to each region, removes the confounds, and hands back a plain 2D array of timepoints by regions.

This is the step where fMRI stops being images and becomes the kind of data you already know how to handle.

In [ ]:
from nilearn.maskers import NiftiMapsMasker

masker = NiftiMapsMasker(maps_img=atlas.maps, standardize="zscore_sample", verbose=0)
time_series = masker.fit_transform(dev.func[0], confounds=dev.confounds[0])

print("shape:", time_series.shape, "= timepoints x regions")
print("this scan sampled every 2 s, so that is",
      time_series.shape[0] * 2, "seconds of data")

168 rows and 39 columns. A NumPy array, exactly like the ones from session 5.

If the download ever fails on you, the same array is saved in this repo at `data/one_subject_timeseries.csv` and `pd.read_csv` will get you back to this point.

## Looking at real signals

In [ ]:
import matplotlib.pyplot as plt

tr = 2.0                                   # seconds per volume
t = np.arange(time_series.shape[0]) * tr
show = ["L DMN", "Med DMN", "Front DMN", "Striate", "L Aud"]

fig, ax = plt.subplots(figsize=(12, 5))
for i, name in enumerate(show):
    j = labels.index(name)
    ax.plot(t, time_series[:, j] + i * 5, linewidth=1.2, label=name)

ax.set_yticks([i * 5 for i in range(len(show))])
ax.set_yticklabels(show, fontsize=12)
ax.set_xlabel("time (s)", fontsize=13)
ax.set_title("Five real brain regions, one participant watching a film", fontsize=15)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

Look at the three default mode regions in the lower part of the figure. They rise and fall together, without us having done anything to make that happen. That co-fluctuation is functional connectivity, and next session we measure it.

The signals are z-scored, so the units are standard deviations rather than anything physical. Raw fMRI intensity has no meaningful scale, which is why nobody reports it.

## A first check

In [ ]:
from scipy.stats import pearsonr

a = time_series[:, labels.index("L DMN")]
b = time_series[:, labels.index("Med DMN")]
c = time_series[:, labels.index("Striate")]

r_within, p_within = pearsonr(a, b)
r_across, p_across = pearsonr(a, c)

print(f"L DMN with Med DMN (same network):     r = {r_within:.2f}, p = {p_within:.4f}")
print(f"L DMN with Striate (different system): r = {r_across:.2f}, p = {p_across:.4f}")

Two regions of the same network correlate strongly. A default mode region and a visual region do not. That is the expected result, and getting the expected result on data you did not simulate is the moment the pipeline earns your trust.

## Try it yourself

Pick two other regions from `labels` and correlate them. `atlas.networks` tells you which system each region belongs to, so you can predict the answer before you run it.

**Next session:** fifty participants, a network each, and a real developmental result.